# 03 · Analysis, figures and app assets

Reads the results saved by notebook 02 and produces the tables, plots and figures used in the README.
Most sections need only Drive and run on a **CPU runtime**:

| Section | Needs |
|---|---|
| 1. Summary tables | CPU, no dataset |
| 2. Summary plots | CPU, no dataset |
| 3. Thresholded metrics (v1 and v2) and calibration shift | CPU, no dataset |
| 4. Border check | CPU, no dataset |
| 5. Per-defect-type AUROC, error analysis, gallery | CPU **and the dataset** (images are drawn) |
| 6. Speed benchmark | **GPU** and the dataset |
| 7. App inference check | CPU, no dataset |
| 8. Bundle for the repo's `docs/` folder | CPU |

The setup cell below does not copy the dataset; section 5 and 6 call `ensure_dataset()` when they need it.
Do not copy numbers from these outputs into the README without checking them against the saved CSVs.

In [ ]:
import logging, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

NEED_DATA = False   # False: most sections only read saved results; sections 5 and 6 call ensure_dataset()

drive.mount("/content/drive")

REPO_URL = "https://github.com/tsegaeph/industrial-anomaly-detection.git"
REPO_DIR = Path("/content/repo")
DRIVE_PROJECT = Path("/content/drive/MyDrive/industrial-anomaly-detection")
DRIVE_DATA = DRIVE_PROJECT / "datasets" / "mvtec_anomaly_detection"
ARCHIVE = DRIVE_PROJECT / "datasets" / "mvtec_7cat.tar"
DATA_ROOT = Path("/content/industrial-anomaly-detection/datasets/mvtec_anomaly_detection")
EXP = DRIVE_PROJECT / "experiments"
REPORTS = DRIVE_PROJECT / "reports"
CATS = ("bottle", "capsule", "carpet", "metal_nut", "screw", "toothbrush", "zipper")


def sh(*cmd) -> None:
    """Run a command, print its output, raise on failure."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode:
        raise RuntimeError(f"{cmd[0]} failed: {r.stderr.strip()}")


# Code always comes from GitHub. After a pull that changes modules you already imported,
# use Runtime -> Restart session (this keeps /content, so the dataset stays).
if REPO_DIR.exists():
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
sys.path.insert(0, str(REPO_DIR))


def dataset_ok() -> bool:
    return all((DATA_ROOT / c / "train" / "good").is_dir() for c in CATS)


def ensure_dataset() -> None:
    """Restore the dataset on Colab's local disk if it is missing or incomplete."""
    if dataset_ok():
        print("Dataset already complete on local disk.")
        return
    shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)          # remove any partial copy
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if ARCHIVE.exists():                       # one large file: much faster than thousands of small ones
        sh("cp", ARCHIVE, "/content/mvtec.tar")
        sh("tar", "-xf", "/content/mvtec.tar", "-C", DATA_ROOT.parent)
        Path("/content/mvtec.tar").unlink(missing_ok=True)
    if not dataset_ok():                       # no archive, or extraction failed
        print("Archive unavailable or incomplete, copying file by file (slow)...")
        shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)
        DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DRIVE_DATA, DATA_ROOT)
    assert dataset_ok(), "Dataset incomplete after copy"
    print("Dataset ready.")


if NEED_DATA:
    ensure_dataset()

logging.basicConfig(level=logging.INFO)

In [ ]:
import numpy as np
import pandas as pd
from src.data.mvtec import CATEGORIES

MET_DIR = REPORTS / "metrics"; MET_DIR.mkdir(parents=True, exist_ok=True)
FIG = REPORTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
METRICS = ["image_auroc", "image_ap", "pixel_auroc", "pixel_ap", "aupro"]
final = EXP / "patchcore_wrn50_256_final"
pc_dir, pm_dir = EXP / "patchcore_wrn50_256", EXP / "padim_r18_256"

## 1. Summary tables

Macro means are the **mean of the seven per-category metrics** (the headline numbers). Pooling all pixels or images
from all categories would let large categories dominate and mix score scales that are not comparable across categories.
Few-shot rows are mean ± std over 3 seeds; the std reflects which normals were sampled (the test set is fixed).

In [ ]:
def macro_row(name):
    p = EXP / name / "results.csv"
    if not p.exists():
        return None
    t = pd.read_csv(p)
    return t[t.category == "MACRO_MEAN"].iloc[0]


def per_cat(name):
    return pd.read_csv(EXP / name / "results.csv").set_index("category")


# A: macro comparison of all configurations
names = ["padim_r18_256", "patchcore_wrn50_256", "patchcore_wrn50_384", "patchcore_wrn50_256_cs1", "patchcore_r18_256"]
A = pd.DataFrame([{"experiment": n, **macro_row(n)[METRICS + ["fit_s", "peak_gpu_gb"]].to_dict()}
                  for n in names if macro_row(n) is not None])
A.to_csv(MET_DIR / "macro_comparison.csv", index=False)
print(A.round(4).to_string(index=False))

# B: resolution effect per category
b = pd.concat({"256": per_cat("patchcore_wrn50_256")[["image_auroc", "pixel_ap", "aupro"]],
               "384": per_cat("patchcore_wrn50_384")[["image_auroc", "pixel_ap", "aupro"]]}, axis=1)
b.to_csv(MET_DIR / "resolution_per_category.csv")
print(b.round(4))

# C: few-shot
rows = []
for n in (5, 10, 25, 50):
    runs = pd.DataFrame([r for r in (macro_row(f"patchcore_n{n}_s{s}") for s in range(3)) if r is not None])
    if len(runs):
        rows.append({"n_train": n, "runs": len(runs),
                     **{m: f"{runs[m].mean():.3f} ± {runs[m].std():.3f}" for m in METRICS}})
full = macro_row("patchcore_wrn50_256")
rows.append({"n_train": "all (1 run)", "runs": 1, **{m: f"{full[m]:.3f}" for m in METRICS}})
C = pd.DataFrame(rows)
C.to_csv(MET_DIR / "few_shot.csv", index=False)
print(C.to_string(index=False))

## 2. Summary plots

Robustness curves, few-shot curve and per-category model comparison (saved to `reports/figures/`).

In [ ]:
from src.visualization.plots import plot_robustness, plot_fewshot, plot_model_comparison

plot_robustness(REPORTS / "metrics/robustness/robustness_macro.csv", FIG / "robustness.png")
plot_fewshot(EXP, FIG / "fewshot.png")
plot_model_comparison(EXP, {"PaDiM R18": "padim_r18_256", "PatchCore WRN50": "patchcore_wrn50_256",
                            "PatchCore WRN50 384px": "patchcore_wrn50_384"}, FIG / "model_comparison.png")

## 3. Thresholded metrics and calibration

Rules (all fixed before looking at the corresponding test results): `max_heldout`, `p99` (99th percentile) and
`mean_plus_3std` of normal scores. Predicted defective when score > threshold.
`oracle_f1` is the best F1 over all thresholds chosen **on the test set**, an upper bound and not a result.
The test set is 78% defective (622 of 800), so precision is flattering: read recall and FPR next to it.

**v1** uses the 10% hold-out scores. (`p99` over only 6 to 32 held-out scores is not meaningful; v1 was run with
`max_heldout` and `mean_plus_3std`.) **v2** uses 5-fold out-of-fold scores; `p99` is the app's default rule.

In [ ]:
from src.evaluation.thresholds import evaluate_thresholds

show = ["category", "threshold", "precision", "recall", "f1", "fpr", "tp", "fp", "fn", "tn", "oracle_f1"]
for tag, calib_file, csv_name in (("v1 held-out", "calibration_heldout_scores.json", "thresholded_metrics.csv"),
                                  ("v2 5-fold CV", "calibration_cv_scores.json", "thresholded_metrics_cv.csv")):
    T = evaluate_thresholds(final, CATEGORIES, calib_file=calib_file)
    T.to_csv(MET_DIR / csv_name, index=False)
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
        for rule, g in T.groupby("rule"):
            print(f"\n== {tag} | rule: {rule} ==")
            print(g[show].to_string(index=False))
            print("macro:", g[["precision", "recall", "f1", "fpr", "oracle_f1"]].mean().round(3).to_dict())

### Calibration shift check

Compares held-out normal scores with test normal scores. In carpet and toothbrush the test normals scored higher
than the held-out ones (carpet: 46% of test normals above the held-out maximum; the calibration models saw only 90%
of the training data, which should make held-out scores higher, not lower). Cause not established.

In [ ]:
import json
from src.evaluation.analysis import load_maps
from src.evaluation.metrics import image_scores

calib = json.loads((final / "calibration_heldout_scores.json").read_text())
for cat in ("carpet", "toothbrush", "screw", "bottle"):
    d = load_maps(final, cat)
    s = image_scores(d["maps"].astype(np.float32))
    test_norm = s[d["labels"] == 0]
    held = np.array(calib[cat]["scores"])
    print(f"{cat:10s} held-out (n={len(held):2d}): min {held.min():.3f} mean {held.mean():.3f} max {held.max():.3f}"
          f" | test normal (n={len(test_norm):2d}): min {test_norm.min():.3f} mean {test_norm.mean():.3f} max {test_norm.max():.3f}"
          f" | frac test normals > held-out max: {(test_norm > held.max()).mean():.2f}")

## 4. Border check

Does the anomaly-map peak sit near the image border (a padding or blur edge effect)? A uniformly random peak lands in
the outer 5% border about 19% of the time. Measured peaks of normal images almost never did (0.00 in six categories,
0.08 in toothbrush, which has only 12 normal test images), so there is no sign of an edge artefact.

In [ ]:
for cat in CATEGORIES:
    d = load_maps(pc_dir, cat)
    m, y = d["maps"].astype(np.float32), d["labels"]
    S = m.shape[-1]; b = int(0.05 * S)
    r, c = divmod(m.reshape(len(m), -1).argmax(1), S)
    near = (r < b) | (r >= S - b) | (c < b) | (c >= S - b)
    print(f"{cat:11s} peak in outer 5% border: normal {near[y == 0].mean():.2f} | defective {near[y == 1].mean():.2f}")

## 5. Error analysis and gallery (needs the dataset)

`per_defect_type_auroc` scores each defect type against all normal test images of its category (from float16-stored
maps, so values can differ slightly from `results.csv`). `ranked_errors` is threshold-free: the lowest-scoring
defective images (likely misses) and the highest-scoring normal images (likely false alarms). In the error figures
the first three rows are misses, the last three are false alarms. The gallery shows seeded random picks (one normal
and two defect types per category), **not cherry-picked**.

In [ ]:
ensure_dataset()

from src.data.mvtec import build_index
from src.evaluation.analysis import per_defect_type_auroc, ranked_errors
from src.visualization.heatmaps import show_cases

df = build_index(DATA_ROOT)

t = per_defect_type_auroc(pc_dir, CATEGORIES).merge(
    per_defect_type_auroc(pm_dir, CATEGORIES), on=["category", "defect_type", "n"], suffixes=("_patchcore", "_padim"))
t.to_csv(MET_DIR / "per_defect_type_auroc.csv", index=False)
print(t.round(3).to_string(index=False))

for cat in ("screw", "toothbrush", "capsule"):
    missed, alarms = ranked_errors(pc_dir, cat, k=3)
    print(f"\n{cat}: lowest-scoring defective images\n", missed[["defect_type", "score"]])
    print(f"{cat}: highest-scoring normal images\n", alarms[["score"]])
    show_cases(pc_dir, df, cat, list(missed.image_path) + list(alarms.image_path),
               save_path=FIG / f"errors_patchcore_{cat}.png")

In [ ]:
rng = np.random.default_rng(0)
for cat in CATEGORIES:
    d = load_maps(final, cat)
    paths, labels, types = d["image_paths"].astype(str), d["labels"], d["defect_types"].astype(str)
    picks = [str(rng.choice(paths[labels == 0]))]
    dtypes = sorted(set(types[labels == 1]))
    for dt in rng.choice(dtypes, size=min(2, len(dtypes)), replace=False):
        picks.append(str(rng.choice(paths[types == dt])))
    show_cases(final, df, cat, picks, save_path=FIG / f"gallery_{cat}.png")

## 6. Speed and memory benchmark (needs a GPU)

Inference only (images preloaded, warm-up excluded), measured on bottle at 256 px. PatchCore's search cost grows with
the memory-bank size, so other categories can differ. `ms_per_img_e2e` in `results.csv` is **not** model speed: it
includes decoding 1024 px PNGs.

In [ ]:
import gc, time, torch
ensure_dataset()
from src.data.mvtec import build_index
from src.data.dataset import MVTecDataset, get_split, make_loader
from src.models.padim import PaDiM
from src.models.patchcore import PatchCore

df = build_index(DATA_ROOT)
train_df, test_df = get_split(df, "bottle", "train"), get_split(df, "bottle", "test")
ds = MVTecDataset(test_df.head(32), 256)
x = torch.stack([ds[i]["image"] for i in range(32)])


def bench(model, batch_size, reps=3):
    for _ in range(2):
        model.predict(x[:batch_size])                       # warm-up
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(reps):
        for i in range(0, len(x), batch_size):
            model.predict(x[i:i + batch_size])
    torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / (reps * len(x)), torch.cuda.max_memory_allocated() / 1e9


for name, make in {"padim_r18": lambda: PaDiM(image_size=256),
                   "patchcore_wrn50": lambda: PatchCore(image_size=256)}.items():
    gc.collect(); torch.cuda.empty_cache()
    model = make().fit(make_loader(train_df, 256, 8))
    for bs in (1, 8):
        ms, peak = bench(model, bs)
        print(f"{name:16s} batch={bs}: {ms:6.1f} ms/img | peak GPU {peak:.2f} GB")
    del model

## 7. App inference check (CPU)

Runs the same `AnomalyPredictor` the Streamlit app uses, on the sample images shipped in `app/samples/`.
One image per case only shows that the code path works: do not quote these verdicts as results. The second call per
image is the inference time (the first includes loading the model). Measured on a Colab CPU runtime with 1 thread:
about 1.4 to 2.9 s per image. The live app is at
`https://industrial-anomaly-detection-te.streamlit.app/`.

In [ ]:
import time, torch
from PIL import Image
from src.inference.predictor import AnomalyPredictor

print("CPU threads:", torch.get_num_threads())
p = AnomalyPredictor(final, REPO_DIR / "app/thresholds.json", rule="p99")
for cat in ("bottle", "screw", "carpet"):
    for kind in ("good", "defect"):
        img = np.asarray(Image.open(REPO_DIR / f"app/samples/{cat}_{kind}.png").convert("RGB"))
        t = time.time(); p.predict(img, cat); first = time.time() - t       # includes model load
        t = time.time(); pred = p.predict(img, cat); second = time.time() - t
        print(f"{cat:8s} {kind:7s} score {pred.score:.3f} thr {pred.threshold:.3f} "
              f"{'DEFECTIVE' if pred.is_anomalous else 'normal   '} | first {first:5.1f}s | second {second:5.1f}s")

## 8. Bundle for the repo

Collects the small CSVs, configs and figures into one zip (skips anything missing and says so). Unzip it into the
repo root on your PC; it creates `docs/figures`, `docs/results` and `docs/dataset_audit`.

In [ ]:
from google.colab import files

OUT = Path("/content/docs_bundle/docs"); shutil.rmtree(OUT.parent, ignore_errors=True)


def put(src, dst_rel):
    src = Path(src)
    if src.exists():
        dst = OUT / dst_rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, dst)
    else:
        print("missing:", src)


for p_ in (REPORTS / "figures").glob("*.png"):
    put(p_, f"figures/{p_.name}")
for p_ in REPORTS.glob("dataset_audit/*"):
    put(p_, f"dataset_audit/{p_.name}")
for p_ in REPORTS.glob("metrics/*.csv"):
    put(p_, f"results/{p_.name}")
for p_ in REPORTS.glob("metrics/robustness/*.csv"):
    put(p_, f"results/robustness/{p_.name}")
for e in ("padim_r18_256", "patchcore_wrn50_256", "patchcore_wrn50_384", "patchcore_wrn50_256_cs1",
          "patchcore_r18_256", "patchcore_wrn50_256_final"):
    put(EXP / e / "results.csv", f"results/experiments/{e}.csv")
    put(EXP / e / "config.yaml", f"results/experiments/{e}.yaml")
put(final / "calibration_cv_scores.json", "results/calibration_cv_scores.json")
put(final / "calibration_heldout_scores.json", "results/calibration_heldout_scores.json")

shutil.make_archive("/content/docs_bundle", "zip", "/content/docs_bundle")
files.download("/content/docs_bundle.zip")